# ACE trajectories example


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import importlib

# Define project root (run from MHDTurbPy repository root)
root_dir = str(Path.cwd())

sc_pos_path = Path(root_dir).joinpath("functions", "sc_pos")
sys.path.insert(0, str(sc_pos_path))

import interactive_orbits_timeseries_plus3d as orbits
importlib.reload(orbits)


## Build the 3D trajectory figure


In [ ]:
targets = ["ACE", "WIND", "PSP"]
fig_3d = orbits.build_3d_figure(
    targets=targets,
    start="2021-10-01T00:00:00",
    stop="2021-12-10T00:00:00",
    step="6h",
    frame3d="HCI",
    rss_rsun=20,
)
fig_3d.show()


## Identify best-aligned GSE intervals (new)

The function below applies a first-principles stream-alignment criterion in **GSE**:

- same stream tube => small **transverse (Y-Z)** separation
- finite **X** separation is turned into a convective lag using an assumed constant $V_{sw}$
- score windows of fixed duration and keep the best `N`

It prints the assumptions used so results are interpretable.

In [ ]:
window_hours = 8
top_n = 3
targets = ["ACE", "WIND", "PSP"]

ranked = orbits.find_best_gse_alignment_intervals(
    targets=targets,
    start="2021-10-01T00:00:00",
    stop="2021-12-10T00:00:00",
    step="30min",
    window_hours=window_hours,
    n_best=top_n,
    vsw_kms=420.0,
    min_coverage=0.9,
    overlap=0.5,
    w_perp=0.7,
    w_lag=0.3,
    verbose=True,
)

display(ranked)

figs = orbits.build_best_alignment_3d_figures(
    ranked_windows=ranked,
    targets=targets,
    step="30min",
    gse_axis_units="Re",
)

for i, fig in enumerate(figs, start=1):
    print(f"\n--- Best interval #{i} ---")
    fig.show()

## Find intervals with best stream alignment in GSE

The method below assumes frozen-in radial solar-wind flow and minimizes a physically-motivated metric built from pairwise **cross-flow** and **along-flow** separations.

In [ ]:
best_windows, alignment_figs, all_scores = orbits.build_best_alignment_interval_figures(
    targets=targets,
    start="2021-10-01T00:00:00",
    stop="2021-12-10T00:00:00",
    step="3h",
    window_hours=18,
    top_n=3,
    vsw_kms=400.0,
    along_weight=0.25,
    min_coverage=0.8,
)

display(best_windows)
for fig in alignment_figs:
    fig.show()